# Phinet ⇄ PyTorch — the same operations, checked to 1e-16

In Phient's world a word is a seed of what it relates to, under what conditions. Phinet is the Fourier tensor engine those relations are computed with: complex functions on a closed circle, a differentiable unitary FFT, a Fourier neural operator, holographic binding, sixty-four spectral gates. This notebook is the PyTorch side of every one of those operations, and a check that it agrees with the Rust engine on a reference dump.

`reference.json` is written by `cargo run --release --example torch_reference` in the `phinet` crate. `phinet_torch.py` holds the torch implementations. Both live in `phinet/torch/`.

In [ ]:
# Colab: fetch the two files. Set RAW to your checkout's raw URL, or upload them with the file panel.
import os, urllib.request
RAW = os.environ.get("PHINET_RAW", "")  # e.g. https://raw.githubusercontent.com/gemphi/phinet/main/torch/
for f in ("phinet_torch.py", "reference.json"):
    if not os.path.exists(f) and RAW:
        urllib.request.urlretrieve(RAW + f, f)
print([f for f in ("phinet_torch.py", "reference.json") if os.path.exists(f)])

In [ ]:
import torch, phinet_torch as pn
torch.set_default_dtype(torch.float64)
fails = pn.check("reference.json")   # 55 checks: every op, every gradient, one Adam step

## The operator, trained in torch

The sixty-second example from the phinet README, in PyTorch: sixty items on a 64-point circle, each item's target is itself rotated by one sample — a relation the identity cannot express and a spectral filter can. Same `Fno` layout as the Rust one, so a trained torch operator's parameters can be written back with `Fno::from_bytes` after a small serialiser.

In [ ]:
import torch, phinet_torch as pn
space, n = 64, 60
g = torch.Generator().manual_seed(0)
def unit_random():
    z = torch.complex(torch.randn(space, generator=g), torch.randn(space, generator=g)); return z / z.abs().pow(2).sum().sqrt()
inputs = torch.stack([unit_random() for _ in range(n)])[:, None, :]          # [n,1,space]
targets = torch.roll(inputs, 1, dims=-1)                                      # rotate by one sample
fno = pn.Fno(space=space, width=4, layers=1, modes=32)
with torch.no_grad():
    for p in fno.parameters(): p.copy_(torch.complex(torch.randn(p.shape, generator=g), torch.randn(p.shape, generator=g)) * (0.25 if p.dim()==3 and p.shape[-1]==space else 0.5))
opt = torch.optim.Adam(fno.parameters(), lr=1e-2)
def mrr():
    with torch.no_grad():
        pred = fno(inputs); cos = pn.cosine_table(pred, targets)[:, :, 0]        # [n, n]
        ranks = (cos > cos.diag()[:, None]).sum(1) + 1
        return (1.0 / ranks).mean().item()
print("untrained MRR", round(mrr(), 3))
for epoch in range(60):
    perm = torch.randperm(n, generator=g)
    for i in range(0, n, 20):
        idx = perm[i:i+20]; decoy = (idx + torch.randint(1, n, (len(idx),), generator=g)) % n
        pred = fno(inputs[idx])
        loss = pn.hinge(pn.cosine(pred, targets[idx]), pn.cosine(pred, targets[decoy]), 0.5)
        opt.zero_grad(); loss.backward(); opt.step(); fno.mask_modes()
print("trained MRR", round(mrr(), 3), "(phinet's README asserts > 0.9 on the same task)")

## The sixty-four gates

A hexagram's six lines gate the six octaves of a function's spectrum. ䷀ (111111) is the identity; ䷁ (000000) keeps only the mean. `novelty` says how much a gate changes a function.

In [ ]:
f = inputs[0, 0]
for hid in (0b111111, 0b101010, 0b000111, 0b000000):
    print(f"{hid:06b}  lines={['%.2f' % v for v in pn.lines(f)]}  novelty={pn.novelty(f, hid):.3f}  hexagram_of={pn.hexagram_of(pn.hexagram_apply(f, hid))}")